# Phase 4: Sequence Clustering and Out-of-Distribution Splits

Phase 2b showed topology beats the burial confound, but with one open caveat:
`GroupKFold` on `pdb_id` prevents *entry* leakage, not **homology** leakage.
Several of the 32 starter structures look to be protein kinases, which share both
fold and ATP-pocket architecture. Splitting by entry can put 1HCK in train and
1KE5 in test, different PDB codes, near-identical binding site.

This notebook builds real out-of-distribution splits and then **re-runs the
Phase 2b probe under them**. If the topology gain survives, it is trustworthy.
If it shrinks, we learn the earlier number was inflated, which is exactly what
this phase exists to find out.

---

### Method

1. **All-versus-all local alignment** (Biopython `PairwiseAligner`, BLOSUM62,
   BLAST-style affine gaps). At 41 chains that is 820 pairs, exact and fully
   transparent, no heuristic needed.
2. **Single-linkage clustering at 30 % identity.** Single-linkage is the
   conservative choice: it merges a cluster if *any* member is similar, so it
   over-merges rather than under-merges. For leakage control, over-merging is
   the safe direction to err in.
3. **MMseqs2 cross-check.** The pipeline needs MMseqs2 at BioLiP scale
   (thousands of chains), where all-versus-all becomes infeasible. It is run here
   on the same input and the two clusterings are compared, so the fast path is
   validated against the exact one *before* being relied on.
4. **Cluster-grouped CV + a fixed train/val/test split** saved for Phase 5.

## Cell 1: Setup and load Phase 2b

In [ ]:
!pip install -q torch-geometric==2.6.1 biopython==1.84

In [ ]:
import os, subprocess, time, warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch

warnings.filterwarnings("ignore")


def locate(fname: str) -> Path:
    cands = []
    for root in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if root.exists():
            cands.extend(root.rglob(fname))
    if not cands:
        raise FileNotFoundError(f"{fname} not found -- attach the producing notebook's output")
    return max(cands, key=lambda p: p.stat().st_size)


blob = torch.load(locate("phase2b_graphs.pt"), map_location="cpu", weights_only=False)
dataset = blob["dataset"]
topo_names = blob["topo_names"]
burial_names = blob["burial_names"]

chain_ids = [f"{d.pdb_id}_{d.chain_id}" for d in dataset]
seqs = {cid: d.seq for cid, d in zip(chain_ids, dataset)}

print(f"chains {len(dataset)} | residues {sum(d.num_nodes for d in dataset):,} "
      f"| node dim {dataset[0].x.size(1)} | topo dim {len(topo_names)}")
print(f"sequence lengths: min {min(len(s) for s in seqs.values())}, "
      f"max {max(len(s) for s in seqs.values())}")

## Cell 2: Exact all-versus-all sequence identity

Identity is computed as *identical aligned columns / length of the shorter
sequence*. That denominator is deliberate: it makes a short domain matching a
long protein score high, which merges more aggressively and therefore splits more
strictly. For leakage control, the conservative direction is the correct one.

In [ ]:
from itertools import combinations

from Bio.Align import PairwiseAligner, substitution_matrices
from joblib import Parallel, delayed

aligner = PairwiseAligner()
aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
aligner.mode = "local"
aligner.open_gap_score = -11        # BLAST defaults
aligner.extend_gap_score = -1


def identity(a: str, b: str) -> float:
    """Fraction identical over the shorter sequence, from the best local alignment."""
    aln = aligner.align(a, b)[0]
    matches = 0
    for (s1, e1), (s2, e2) in zip(*aln.aligned):
        matches += sum(1 for x, y in zip(a[s1:e1], b[s2:e2]) if x == y)
    return matches / max(1, min(len(a), len(b)))


pairs = list(combinations(chain_ids, 2))
t0 = time.time()
vals = Parallel(n_jobs=max(1, os.cpu_count() or 2))(
    delayed(identity)(seqs[i], seqs[j]) for i, j in pairs)
print(f"{len(pairs)} pairwise alignments in {time.time() - t0:.1f}s")

idx = {c: i for i, c in enumerate(chain_ids)}
ident = np.eye(len(chain_ids), dtype=np.float32)
for (i, j), v in zip(pairs, vals):
    ident[idx[i], idx[j]] = ident[idx[j], idx[i]] = v

hi = [(v, i, j) for (i, j), v in zip(pairs, vals) if v >= 0.30]
hi.sort(reverse=True)
print(f"\npairs at >=30% identity: {len(hi)}")
for v, i, j in hi[:12]:
    print(f"   {i:<10} {j:<10} {v:.3f}")

## Cell 3: Single-linkage clustering at 30 % identity

Also prints cluster membership, which directly tests the kinase hypothesis from
Phase 2b: if those entries land in one cluster, entry-level splits were leaking.

In [ ]:
IDENTITY_THRESHOLD = 0.30

parent = list(range(len(chain_ids)))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

for i in range(len(chain_ids)):
    for j in range(i + 1, len(chain_ids)):
        if ident[i, j] >= IDENTITY_THRESHOLD:
            union(i, j)

clusters = defaultdict(list)
for i, c in enumerate(chain_ids):
    clusters[find(i)].append(c)
clusters = {f"C{k:02d}": v for k, v in enumerate(sorted(clusters.values(), key=len, reverse=True))}
chain_to_cluster = {c: k for k, v in clusters.items() for c in v}

print(f"{len(clusters)} clusters from {len(chain_ids)} chains "
      f"at {IDENTITY_THRESHOLD:.0%} identity\n")
for k, v in clusters.items():
    if len(v) > 1:
        print(f"  {k} (n={len(v)}): {', '.join(sorted(v))}")
singletons = [k for k, v in clusters.items() if len(v) == 1]
print(f"  ...plus {len(singletons)} singleton cluster(s)")

# --- direct test of the Phase 2b leakage caveat ------------------------------
KINASE_GUESS = {"1ATP", "1HCK", "1KE5", "1M17", "1QPE", "1PHK", "1XKK", "2SRC"}
present = {c for c in chain_ids if c.split("_")[0] in KINASE_GUESS}
kin_clusters = {chain_to_cluster[c] for c in present}
print(f"\nsuspected kinase chains: {len(present)} -> {len(kin_clusters)} cluster(s): "
      f"{sorted(kin_clusters)}")
print("  entry-level splits WERE leaking homologues" if len(kin_clusters) < len(present)
      else "  no merging at this threshold -- entry-level split was not leaking via kinases")

## Cell 4: MMseqs2 cross-check

MMseqs2 is what this pipeline must use once Phase 4 scales to thousands of
BioLiP chains, where 820 exact alignments becomes tens of millions. Validating it
against the exact clustering *now*, while ground truth is still cheap to compute,
is the point. If the binary cannot be fetched the notebook continues on the exact
clustering, this cell is a check, not a dependency.

In [ ]:
MMSEQS_URL = "https://mmseqs.com/latest/mmseqs-linux-avx2.tar.gz"
mmseqs_bin = None
try:
    subprocess.run(f"curl -sL {MMSEQS_URL} -o /tmp/mm.tar.gz && tar xzf /tmp/mm.tar.gz -C /tmp",
                   shell=True, check=True, timeout=300)
    cand = Path("/tmp/mmseqs/bin/mmseqs")
    if cand.exists():
        os.chmod(cand, 0o755)
        subprocess.run([str(cand), "version"], check=True, capture_output=True, timeout=60)
        mmseqs_bin = str(cand)
        print(f"mmseqs2 ready: {mmseqs_bin}")
except Exception as e:
    print(f"mmseqs2 unavailable ({type(e).__name__}) -- continuing with exact clustering")

mm_clusters = None
if mmseqs_bin:
    work = Path("/tmp/mmwork"); work.mkdir(exist_ok=True)
    fasta = work / "in.fasta"
    fasta.write_text("".join(f">{c}\n{seqs[c]}\n" for c in chain_ids))
    try:
        subprocess.run(
            [mmseqs_bin, "easy-cluster", str(fasta), str(work / "res"), str(work / "tmp"),
             "--min-seq-id", str(IDENTITY_THRESHOLD), "-c", "0.8", "--cov-mode", "1",
             "-v", "1"],
            check=True, capture_output=True, timeout=600)
        rep = defaultdict(list)
        for line in (work / "res_cluster.tsv").read_text().splitlines():
            r, m = line.split("\t")
            rep[r].append(m)
        mm_clusters = {f"M{i:02d}": v for i, v in
                       enumerate(sorted(rep.values(), key=len, reverse=True))}
        print(f"mmseqs2: {len(mm_clusters)} clusters (exact method: {len(clusters)})")
    except subprocess.CalledProcessError as e:
        print(f"mmseqs2 run failed: {e.stderr.decode()[:300]}")

if mm_clusters:
    # Agreement = do both methods make the same same-cluster/different-cluster call?
    mm_of = {c: k for k, v in mm_clusters.items() for c in v}
    agree = sum(1 for i, j in pairs
                if (chain_to_cluster[i] == chain_to_cluster[j]) == (mm_of[i] == mm_of[j]))
    print(f"pairwise agreement with exact clustering: {agree}/{len(pairs)} "
          f"= {100 * agree / len(pairs):.1f}%")
    disagree = [(i, j, ident[idx[i], idx[j]]) for i, j in pairs
                if (chain_to_cluster[i] == chain_to_cluster[j]) != (mm_of[i] == mm_of[j])]
    for i, j, v in sorted(disagree, key=lambda t: -t[2])[:8]:
        print(f"   disagree: {i:<10} {j:<10} exact identity {v:.3f}")

## Cell 5: Re-run the Phase 2b probe under cluster splits

Identical models and feature sets to Phase 2b. The only change is the grouping
variable: `pdb_id` → `cluster`. Any drop in the topology gain is the size of the
homology leakage that was inflating the earlier result.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

N_SEQ = 27                                   # Phase 1 feature width
topo_raw = np.concatenate([d.x_topo.numpy() for d in dataset], axis=0)
seq_feats = np.concatenate([d.x[:, :N_SEQ].numpy() for d in dataset], axis=0)
y = np.concatenate([d.y.numpy() for d in dataset]).astype(int)
g_entry = np.concatenate([[d.pdb_id] * d.num_nodes for d in dataset])
g_clust = np.concatenate([[chain_to_cluster[c]] * d.num_nodes
                          for c, d in zip(chain_ids, dataset)])

b_idx = [topo_names.index(n) for n in burial_names]
t_idx = [i for i in range(len(topo_names)) if i not in b_idx]

SETS = {
    "seq":             seq_feats,
    "burial":          topo_raw[:, b_idx],
    "topo":            topo_raw[:, t_idx],
    "seq+burial":      np.hstack([seq_feats, topo_raw[:, b_idx]]),
    "seq+burial+topo": np.hstack([seq_feats, topo_raw]),
}


def evaluate(X, groups, model, n_splits=5):
    roc, pr = [], []
    for tr, te in GroupKFold(n_splits=n_splits).split(X, y, groups):
        clf = (make_pipeline(StandardScaler(),
                             LogisticRegression(max_iter=3000, class_weight="balanced"))
               if model == "logreg" else
               HistGradientBoostingClassifier(max_iter=200, learning_rate=0.08,
                                              max_depth=6, random_state=0))
        clf.fit(X[tr], y[tr])
        p = clf.predict_proba(X[te])[:, 1]
        roc.append(roc_auc_score(y[te], p)); pr.append(average_precision_score(y[te], p))
    return np.array(roc), np.array(pr)


n_splits = min(5, len(clusters))
print(f"grouping by cluster ({len(clusters)} groups, {n_splits}-fold)\n")

res = {}
for grouping, groups in (("entry", g_entry), ("cluster", g_clust)):
    for model in ("logreg", "hgb"):
        print(f"=== {model} | grouped by {grouping} ===")
        print(f"{'feature set':<18}{'ROC-AUC':>16}{'PR-AUC':>16}")
        for name, X in SETS.items():
            roc, pr = evaluate(X, groups, model, n_splits)
            res[(grouping, model, name)] = (roc, pr)
            print(f"{name:<18}{roc.mean():>9.3f} +-{roc.std():.3f}"
                  f"{pr.mean():>10.3f} +-{pr.std():.3f}")
        b = res[(grouping, model, "seq+burial")][1]
        f = res[(grouping, model, "seq+burial+topo")][1]
        d = f - b
        print(f"  topology marginal PR-AUC: {d.mean():+.4f} +-{d.std():.4f}  "
              f"({(d > 0).sum()}/{len(d)} folds)\n")

print("=" * 62)
print("LEAKAGE IMPACT -- topology marginal PR-AUC gain, entry vs cluster split")
for model in ("logreg", "hgb"):
    e = (res[("entry", model, "seq+burial+topo")][1]
         - res[("entry", model, "seq+burial")][1]).mean()
    c = (res[("cluster", model, "seq+burial+topo")][1]
         - res[("cluster", model, "seq+burial")][1]).mean()
    print(f"  {model:<8} entry {e:+.4f}   cluster {c:+.4f}   "
          f"shrinkage {100 * (1 - c / e) if abs(e) > 1e-9 else float('nan'):.0f}%")

## Cell 6: Fixed train/val/test split for Phase 5

Whole clusters are assigned to one side, largest-first, greedily balancing the
**positive-residue** count rather than the chain count, with 41 chains, an
unlucky split could otherwise leave a fold with almost no binding-site residues
and make the metrics meaningless.

In [ ]:
TARGET = {"train": 0.70, "val": 0.15, "test": 0.15}

pos_per_chain = {c: int(d.y.sum()) for c, d in zip(chain_ids, dataset)}
cluster_pos = {k: sum(pos_per_chain[c] for c in v) for k, v in clusters.items()}
total_pos = sum(cluster_pos.values())

assign, got = {}, {k: 0 for k in TARGET}
for k in sorted(cluster_pos, key=lambda x: -cluster_pos[x]):
    split = min(TARGET, key=lambda s: got[s] / max(TARGET[s], 1e-9))
    assign[k] = split
    got[split] += cluster_pos[k]

chain_split = {c: assign[chain_to_cluster[c]] for c in chain_ids}
for d, c in zip(dataset, chain_ids):
    d.split = chain_split[c]
    d.cluster = chain_to_cluster[c]

print(f"{'split':<8}{'clusters':>10}{'chains':>9}{'residues':>11}{'positives':>11}{'pos rate':>10}")
for s in TARGET:
    ch = [c for c in chain_ids if chain_split[c] == s]
    n_res = sum(int(d.num_nodes) for d, c in zip(dataset, chain_ids) if chain_split[c] == s)
    n_pos = sum(pos_per_chain[c] for c in ch)
    print(f"{s:<8}{sum(1 for k in assign.values() if k == s):>10}{len(ch):>9}"
          f"{n_res:>11,}{n_pos:>11,}{100 * n_pos / max(n_res, 1):>9.2f}%")

# --- leakage assertion: no cluster may straddle two splits --------------------
for k, members in clusters.items():
    assert len({chain_split[c] for c in members}) == 1, f"cluster {k} straddles splits"
cross = [(i, j, float(ident[idx[i], idx[j]])) for i, j in pairs
         if chain_split[i] != chain_split[j] and ident[idx[i], idx[j]] >= IDENTITY_THRESHOLD]
print(f"\ncross-split pairs >= {IDENTITY_THRESHOLD:.0%} identity: {len(cross)}")
assert not cross, f"LEAKAGE: {cross[:5]}"
print("no homologous pair spans two splits -- split is clean")

out = Path("/kaggle/working/processed"); out.mkdir(parents=True, exist_ok=True)
p = out / "phase4_splits.pt"
torch.save({"dataset": dataset,
            "clusters": clusters, "chain_to_cluster": chain_to_cluster,
            "chain_split": chain_split, "identity_matrix": ident,
            "chain_ids": chain_ids, "identity_threshold": IDENTITY_THRESHOLD,
            "cv_results": {f"{a}|{b}|{c}": (v[0].tolist(), v[1].tolist())
                           for (a, b, c), v in res.items()},
            "topo_names": topo_names, "burial_names": burial_names,
            "node_feat_dim": int(dataset[0].x.size(1)),
            "edge_feat_dim": int(dataset[0].edge_attr.size(1)),
            "pos_weight": blob["pos_weight"], "aa_alphabet": blob["aa_alphabet"]}, p)
print(f"saved -> {p} ({p.stat().st_size / 1e6:.1f} MB)")

## Cell 7: Identity matrix and split visualisation

In [ ]:
import matplotlib.pyplot as plt

order = sorted(range(len(chain_ids)), key=lambda i: (chain_to_cluster[chain_ids[i]], chain_ids[i]))
M = ident[np.ix_(order, order)]

fig, ax = plt.subplots(1, 2, figsize=(12.5, 5))
im = ax[0].imshow(M, cmap="magma", vmin=0, vmax=1)
ax[0].set_title(f"pairwise sequence identity (clustered order)")
ax[0].set_xticks([]); ax[0].set_yticks(range(len(order)))
ax[0].set_yticklabels([chain_ids[i] for i in order], fontsize=5)
plt.colorbar(im, ax=ax[0], fraction=0.046)

names = list(TARGET)
res_counts = [sum(int(d.num_nodes) for d, c in zip(dataset, chain_ids) if chain_split[c] == s)
              for s in names]
pos_counts = [sum(pos_per_chain[c] for c in chain_ids if chain_split[c] == s) for s in names]
x = np.arange(len(names))
ax[1].bar(x - 0.2, res_counts, 0.4, label="residues")
ax[1].bar(x + 0.2, pos_counts, 0.4, label="positives")
ax[1].set_xticks(x); ax[1].set_xticklabels(names); ax[1].set_yscale("log")
ax[1].legend(); ax[1].set_title("split balance (log scale)")
plt.tight_layout(); plt.show()

---

## What this establishes

**The leakage-impact block is the result.** It states, in PR-AUC, how much of
the Phase 2b topology gain was homology leakage:

- **Gain largely holds under cluster splits** → the Phase 2b finding was real and
  the numbers to report are these, not the entry-split ones.
- **Gain shrinks substantially** → the honest headline is the cluster-split value.
  Report it, and note the entry-split figure as the inflated version. Quantifying
  your own leakage is a strength in an ISEF writeup, not a weakness.

`phase4_splits.pt` carries `d.split` and `d.cluster` on every graph, plus the
full identity matrix, so Phase 5 never re-derives them and cannot silently
disagree about the split.

### Caveat that does not go away

41 chains in a handful of clusters is a *small* number of independent units. Fold
variance will be large and no split scheme fixes that, only more data does. The
BioLiP scale-up is what makes these numbers publication-grade; this notebook is
what makes them honest at the current scale.